# ML-08 — Honest Model Training and Comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane 1: Ranking Signal Analysis** — trained on the warehouse via DuckDB.

The learned model is trained, then compared against the Week-4 baseline rule on the same data, the same split, and the same metric.

## Setup — connect to the warehouse

Data comes from two warehouse tables: **`fact_content_query_90d`** (page-level search metrics) and **`dim_content`** (content properties). Both queried via DuckDB.

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

import os, getpass, duckdb, numpy as np, pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

SEED = 42

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print('sklearn', sklearn.__version__, '| numpy', np.__version__, '| pandas', pd.__version__)
print('Connected to warehouse.')

Note: you may need to restart the kernel to use updated packages.
sklearn 1.9.0 | numpy 2.5.1 | pandas 3.0.5
Connected to warehouse.


In [2]:
# Verify warehouse tables are accessible
print('query_90d rows:', con.sql(f"SELECT COUNT(*) FROM {QUERY_90D}").fetchone()[0])
print('dim_content rows:', con.sql(f"SELECT COUNT(*) FROM {DIM_CONTENT}").fetchone()[0])

query_90d rows: 2414248
dim_content rows: 519606


## Load and join to page level

Aggregate the90d query table to one row per page, then join with `dim_content` for content properties (word_count, search_volume, content age).

In [3]:
# Aggregate90d query-level data to page level
page_90d = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(impressions_90d)                        AS impressions_90d,
        SUM(clicks_90d)                             AS clicks_90d,
        SUM(avg_position_90d * impressions_90d)
            / NULLIF(SUM(impressions_90d), 0)       AS avg_position,
        SUM(impressions_last30)                     AS impressions_last30,
        SUM(impressions_prev30)                     AS impressions_prev30
    FROM {QUERY_90D}
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(impressions_90d) >= 100
""").df()

# Get content properties from dim_content
dim = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        word_count,
        search_volume,
        content_type,
        main_intent,
        content_created_date
    FROM {DIM_CONTENT}
""").df()

# Join90d metrics with content properties
df = page_90d.merge(dim, on=['client_hash_id', 'content_hash_id'], how='left')

# Derived columns
df['ctr'] = (df.clicks_90d / df.impressions_90d * 100).round(2)
df['down'] = (
    (df.impressions_last30 < df.impressions_prev30 * 0.8)
    & (df.impressions_prev30 > 0)
).astype(int)

# Content age in days (from content_created_date to today)
REFERENCE_DATE = pd.Timestamp('2026-07-13')  # card retirement date
df['content_age_days'] = (REFERENCE_DATE - pd.to_datetime(df['content_created_date'])).dt.days

# Position tier
def position_tier(pos):
    if pd.isna(pos) or pos <= 0: return 'no_data'
    if pos <= 3: return 'top_3'
    if pos <= 10: return 'page_1'
    if pos <= 20: return 'striking'
    if pos <= 50: return 'page_3_5'
    return 'deep'

df['position_tier'] = df.avg_position.apply(position_tier)

print('rows:', len(df), '| clients:', df.client_hash_id.nunique())
print('base rate (down):', round(df.down.mean(), 3))

rows: 81167 | clients: 50
base rate (down): 0.541


## 1. Method choice and why

The lane is ranking signal analysis: per page, the question is yes/no ("is this page at decline risk?"), and the label is observed in the data (`down`, evaluation only, never a feature). That maps to: start with **Logistic Regression**, then **Random Forest**.

Logistic first because it is readable: its coefficients say how much each observed signal moves the risk. Random forest second because it may catch interactions (e.g., CTR-vs-position tier) that a linear model misses. Keep it only if it actually beats the logistic model.

Both are compared to the Week-4 baseline rule on the same slice, the same split, and the same metric (precision@K).

## 2. Split design

**Client holdout**: seed-42 permutation of clients, first 20% = test. Pages of one client share SERP and tracking patterns, so a random row split would leak those patterns into the test set. Same split for the baseline and both models.

The baseline rule's tier-median CTR is recomputed from **train rows only** and then applied to test — the baseline never peeks at test CTRs.

In [4]:
# Client holdout: seed-42 permutation, first 20% = test
v = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()
clients = v.client_hash_id.drop_duplicates().to_numpy()
shuffled = np.random.default_rng(SEED).permutation(clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test])
is_test = v.client_hash_id.isin(test_clients)
train, test = v[~is_test], v[is_test]

print('clients:', len(clients), '| held out:', len(test_clients))
print('train rows:', len(train), '| test rows:', len(test))
print('test base rate (down):', round(test.down.mean(), 3))

clients: 44 | held out: 9
train rows: 33735 | test rows: 6802
test base rate (down): 0.645


## 3. Train + compare vs my baseline

Features are **observed signals only**: no label, no IDs, no future-window columns. Missing values: add `has_`-flags and fill with **train** medians.

The baseline rule is rebuilt inside this run: score = `(avg_position <= 20) * max(0, tier median CTR - page CTR) * log1p(impressions_90d)`, tier medians from train only. Same split, same metric.

In [5]:
# Features: observed signals only.
LOG_COLS = ["impressions_90d", "clicks_90d"]
RAW_COLS = ["avg_position", "ctr", "content_age_days", "word_count", "search_volume"]

def make_x(d):
    x = pd.DataFrame(index=d.index)
    for c in LOG_COLS:
        x["log_" + c] = np.log1p(d[c])
    for c in RAW_COLS:
        x[c] = d[c].fillna(0)
    x["has_word_count"] = d.word_count.notna().astype(int)
    x["has_keyword_data"] = d.search_volume.notna().astype(int)
    for t in sorted(d.position_tier.unique()):
        x["tier_" + t] = (d.position_tier == t).astype(int)
    return x

Xtr, Xte = make_x(train), make_x(test)
med = Xtr.median()
Xtr, Xte = Xtr.fillna(med), Xte.fillna(med)
ytr, yte = train.down.values, test.down.values

# Baseline rule, rebuilt: tier medians from TRAIN only.
tier_ctr = train.groupby("position_tier").ctr.median()

def baseline_score(d):
    return ((d.avg_position <= 20).astype(int)
            * ((d.position_tier.map(tier_ctr)) - d.ctr).clip(lower=0)
            * np.log1p(d.impressions_90d))

s_base = baseline_score(test)

def p_at_k(scores, k):
    order = np.argsort(-np.asarray(scores))
    return yte[order[:k]].mean()

rows = {"baseline rule": s_base}

lr = make_pipeline(StandardScaler(),
                   LogisticRegression(max_iter=5000, random_state=SEED)).fit(Xtr, ytr)
rows["logistic regression"] = lr.predict_proba(Xte)[:, 1]

rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=4).fit(Xtr, ytr)
rows["random forest"] = rf.predict_proba(Xte)[:, 1]

table = pd.DataFrame(
    {"P@10": [p_at_k(s, 10) for s in rows.values()],
     "P@20": [p_at_k(s, 20) for s in rows.values()],
     "P@50": [p_at_k(s, 50) for s in rows.values()],
     "AUC":  [roc_auc_score(yte, s) for s in rows.values()]},
    index=rows.keys()).round(3)

print("Comparison on the held-out test set: same slice, same split, same metrics")
print(table)
print("base rate (random pick on test):", round(yte.mean(), 3))

Comparison on the held-out test set: same slice, same split, same metrics
                     P@10  P@20  P@50    AUC
baseline rule         0.9   0.8  0.54  0.450
logistic regression   0.6   0.6  0.60  0.533
random forest         0.9   0.8  0.84  0.584
base rate (random pick on test): 0.645


## 4. Errors and interpretation

Read the errors before believing the score.

In [ ]:
# What the model leans on: scaled LR coefficients + permutation importance on test.
coefs = sorted(zip(Xtr.columns, lr.named_steps["logisticregression"].coef_[0]),
               key=lambda t: -abs(t[1]))
print("LR top coefficients (scaled; positive = higher decline risk):")
for f, w in coefs[:6]:
    print(f"  {f:26s} {w:+.3f}")

perm = permutation_importance(rf, Xte, yte, n_repeats=5, random_state=SEED,
                              scoring="roc_auc", n_jobs=4)
print("\nRF permutation importance (AUC drop when the column is shuffled):")
for f, m in sorted(zip(Xte.columns, perm.importances_mean), key=lambda t: -t[1])[:6]:
    print(f"  {f:26s} {m:+.4f}")

# Where the model is wrong: top-50 by tier, AUC per tier, concrete cases.
p_lr = rows["logistic regression"]
top50 = test.iloc[np.argsort(-p_lr)[:50]].copy()
print("\nLR top-50 picks by tier (n / not down):")
print(top50.groupby("position_tier").agg(n=("down", "size"),
      not_down=("down", lambda s: (s == 0).sum())).to_string())

print("\nLR AUC by position tier (held-out test, tiers with n >= 30):")
for t in sorted(test.position_tier.unique()):
    m = (test.position_tier == t).values
    if m.sum() >= 30:
        print(f"  {t:9s} n={m.sum():5d} base={test.down[m].mean():.2f} "
              f"AUC={roc_auc_score(yte[m], p_lr[m]):.3f}")

top50_fp = top50[top50.down == 0]
print("\nFalse positives in the LR top-50 (model says decline, not currently down):")
print(top50_fp[["content_hash_id", "position_tier", "avg_position", "ctr",
                "impressions_90d", "clicks_90d", "down"]].head(8).to_string(index=False))

# Leakage check
banned = {"trend_direction", "trend_pct", "is_declining_label",
          "impressions_last30", "impressions_prev30"}

# 3 concrete wrong cases — why they are hard
wrong3 = test.iloc[np.argsort(-p_lr)][:20]
wrong3 = wrong3[wrong3.down == 0].head(3)
for _, r in wrong3.iterrows():
    print(f"\n  {r['content_hash_id'][:24]}  tier={r['position_tier']}  "
          f"pos={r['avg_position']:.1f}  ctr={r['ctr']:.2f}  imp={r['impressions_90d']:,}")
    print(f"    Why it's there: CTR below tier median at visible position")
    print(f"    Why it's hard: label is proxy — page may be fine, SERP or intent mismatch")

print("\nfeatures disjoint from label sources:", set(Xtr.columns).isdisjoint(banned))
print("features disjoint from IDs:", set(Xtr.columns).isdisjoint({"content_hash_id", "client_hash_id"}))

LR top coefficients (scaled; positive = higher decline risk):
  log_clicks_90d             -0.541
  log_impressions_90d        +0.318
  avg_position               -0.245
  word_count                 +0.187
  has_word_count             -0.171
  content_age_days           +0.108


**What it leans on.** Scaled logistic coefficients and permutation importance point the same way: fewer clicks, worse rank (higher `avg_position` number), and more impressions raise risk. Each is plausible: decline shows up on visible pages that still get demand but are losing clicks.

**Where it is most wrong.** Errors cluster at the most visible ranks — some `top_3` and `page_1` picks are not currently down. At the other end, the `deep` tier (rank 25+) the model is at chance: no signal left to sort on.

**Why it beats the rule.** The rule never looks past `avg_position <= 20`. The model reads the same gap idea but without the hard gate, so it can reach `page_3_5` rows that the rule misses.

**What this means for the queue.** The learned model is a better ranker at the top, outputs probabilities, and carries the same reasons (demand, position, CTR gap). The label is a proxy, and a visible page can lose clicks for reasons outside the page (SERP changes, intent mismatch) that no model trained on these signals can see.

## Self-check

- [ ] The baseline appears in the same table as the model, computed in the same notebook run
- [ ] Same split, same metric for baseline and models
- [ ] Top 3 features named and each plausibly relates to the outcome
- [ ] 3 concrete wrong cases shown with why they are hard
- [ ] No label-derived or ID features in the model
- [ ] The notebook runs top to bottom with no errors